# Notebook 19b: Supplementary Experiments

## Purpose
Complete the missing experiments from Notebook 19:
1. **Experiment 1**: Regression vs Classification (population data)
2. **Experiment 4**: Function Complexity (Total Variation)
3. **Experiment 5**: Task Difficulty (if not completed)
4. **Frequency Analysis**: FFT of elevation data
5. **Error Analysis**: Spatial patterns in predictions

## What's Missing from NB19
- ❌ Exp 1 failed due to population data path issue
- ❌ Exp 4 failed due to dependency on Exp 1
- ❓ Exp 5 status unclear

## Expected Runtime
~2 hours on Colab T4 GPU

In [10]:
# Setup (same as NB19)
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip gpw_data 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

Cloning into 'satclip'...
remote: Enumerating objects: 584, done.
remote: Counting objects: 100% (263/263), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 584 (delta 194), reused 190 (delta 152), pack-reused 321 (from 2)
Receiving objects: 100% (584/584), 82.79 MiB | 58.67 MiB/s, done.
Resolving deltas: 100% (297/297), done.


In [11]:
from google.colab import drive
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import glob
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score
import positional_encoding as PE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

drive.mount('/content/drive')

Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## Fix: Load Population Data

Correct the data loading from NB19.

In [12]:
print("="*70)
print("LOADING GPW POPULATION DATA")
print("="*70)

GPW_DIR = './gpw_data'
os.makedirs(GPW_DIR, exist_ok=True)

zip_path = '/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip'

if not os.path.exists(zip_path):
    print(f"❌ Zip not found at: {zip_path}")
    print("Please update the path if dataverse_files.zip is in a different location.")
else:
    print(f"✅ Found zip at: {zip_path}")

    # Extract ALL files from zip
    print("\nExtracting all files...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        print(f"Files in zip: {len(z.namelist())}")
        z.extractall(GPW_DIR)
    print("✅ Extraction complete\n")

    # Find all .tif files
    print("Searching for GeoTIFF files...")
    tif_files = glob.glob(f'{GPW_DIR}/**/*.tif', recursive=True)
    print(f"Found {len(tif_files)} .tif files:")
    for f in tif_files:
        print(f"  - {f}")

    # Load population data
    if len(tif_files) > 0:
        # Use first .tif file (or specify exact one if multiple)
        pop_file = tif_files[0]
        print(f"\nLoading: {pop_file}")

        Image.MAX_IMAGE_PIXELS = None
        img = Image.open(pop_file)
        pop_data = np.array(img)
        h, w = pop_data.shape
        lons_pop = np.linspace(-180 + 180/w, 180 - 180/w, w)
        lats_pop = np.linspace(90 - 90/h, -90 + 90/h, h)

        print(f"✅ Population data loaded: {pop_data.shape}")
        print(f"   Value range: [{pop_data.min():.2f}, {pop_data.max():.2f}]")
    else:
        print("❌ No .tif files found in extraction")
        pop_data = None

print("="*70)

LOADING GPW POPULATION DATA
✅ Found zip at: /content/drive/MyDrive/grad/learned_activations/dataverse_files.zip

Extracting all files...
Files in zip: 54
✅ Extraction complete

Searching for GeoTIFF files...
Found 0 .tif files:
❌ No .tif files found in extraction


---
## Model Definitions (Copy from NB19)

In [13]:
# Copy activation classes from NB19

class SplineActivation(nn.Module):
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        return y_low + weight * (y_high - y_low)


class SirenLayer(nn.Module):
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first
        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()

    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0
        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))


class UniversalEncoder(nn.Module):
    def __init__(self, input_type='sh', sh_legendre_polys=10,
                 activation_type='spline', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type

        if input_type == 'raw':
            self.posenc = None
            input_dim = 2
        elif input_type == 'sh':
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
        else:
            raise ValueError(f"Unknown input_type: {input_type}")

        if activation_kwargs is None:
            activation_kwargs = {}

        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None

        elif activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])
            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])
            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)
        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        if self.input_type == 'raw':
            x = coords / torch.tensor([180., 90.], device=coords.device)
        else:
            x = self.posenc(coords)

        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:
            x = self.net(x)

        return x


print("✅ Model classes loaded")

✅ Model classes loaded


---
## Training Utilities (Copy from NB19)

In [14]:
def sample_blocked(data, lons, lats, n_samples=15000, grid_size=5.0, test_ratio=0.3, seed=42):
    np.random.seed(seed)
    valid = data > -1e30
    n_lon = int(360 / grid_size)
    n_lat = int(180 / grid_size)
    n_cells = n_lon * n_lat
    test_cells = set(np.random.choice(n_cells, int(n_cells * test_ratio), replace=False))
    valid_idx = np.where(valid)
    n_valid = len(valid_idx[0])
    sample_idx = np.random.choice(n_valid, min(n_samples, n_valid), replace=False)
    rows, cols = valid_idx[0][sample_idx], valid_idx[1][sample_idx]
    sample_lons, sample_lats = lons[cols], lats[rows]
    sample_vals = data[rows, cols]
    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        cell = int((lat + 90) / grid_size) * n_lon + int((lon + 180) / grid_size)
        cell = min(cell, n_cells - 1)
        train_mask.append(cell not in test_cells)
    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)
    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


class RegressionPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


class ClassificationPredictor(nn.Module):
    def __init__(self, encoder, n_classes=100):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_classes)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords))


def train_regression(name, encoder, coords_train, vals_train, coords_test, vals_test,
                    epochs=100, batch_size=256, lr=1e-3, verbose=False):
    model = RegressionPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(np.log1p(vals_train), dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(np.log1p(vals_test), dtype=torch.float32)

    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)

    best_r2 = -float('inf')
    start = time.time()

    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        if (epoch + 1) % 20 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)
            if verbose:
                print(f"  Epoch {epoch+1}/{epochs}: R² = {r2:.4f}")

    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'model': name,
        'task': 'regression',
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
        'trained_model': model
    }


def train_classification(name, encoder, coords_train, vals_train, coords_test, vals_test,
                        n_bins=100, epochs=100, batch_size=256, lr=1e-3, verbose=False):
    model = ClassificationPredictor(encoder, n_classes=n_bins).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    log_vals = np.log1p(vals_train)
    bins = np.linspace(log_vals.min(), log_vals.max(), n_bins + 1)
    train_labels = np.digitize(log_vals, bins) - 1
    train_labels = np.clip(train_labels, 0, n_bins - 1)

    log_test = np.log1p(vals_test)
    test_labels = np.digitize(log_test, bins) - 1
    test_labels = np.clip(test_labels, 0, n_bins - 1)

    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(train_labels, dtype=torch.long)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(test_labels, dtype=torch.long)

    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)

    best_acc = 0.0
    start = time.time()

    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        if (epoch + 1) % 20 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy().argmax(axis=1)
            acc = (pred == test_y.numpy()).mean()
            best_acc = max(best_acc, acc)
            if verbose:
                print(f"  Epoch {epoch+1}/{epochs}: Acc = {acc:.4f}")

    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'model': name,
        'task': 'classification',
        'accuracy': best_acc,
        'params': n_params,
        'time': train_time,
        'trained_model': model
    }


print("✅ Training utilities loaded")

✅ Training utilities loaded


---
## Experiment 1: Regression vs Classification (CRITICAL)

**Paper's Key Prediction**: Learned activations help regression more than classification.

In [15]:
print("="*80)
print("EXPERIMENT 1: REGRESSION VS CLASSIFICATION (RERUN)")
print("="*80)

if pop_data is not None:
    # Sample population data
    coords_train_pop, vals_train_pop, coords_test_pop, vals_test_pop = sample_blocked(
        pop_data, lons_pop, lats_pop, n_samples=15000
    )
    print(f"\nPopulation data: {len(coords_train_pop)} train, {len(coords_test_pop)} test")

    results_exp1 = []
    activations = ['relu', 'spline', 'siren']

    # Test Regression
    print("\n" + "-"*80)
    print("REGRESSION FORMULATION")
    print("-"*80)

    for act in activations:
        print(f"\nTesting {act.upper()}...")

        if act == 'spline':
            kwargs = {'n_knots': 15, 'init': 'relu'}
        else:
            kwargs = None

        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_regression(
            f'SH + {act.upper()} (reg)', enc,
            coords_train_pop, vals_train_pop,
            coords_test_pop, vals_test_pop,
            verbose=True
        )
        results_exp1.append(res)
        print(f"  Final R²: {res['r2']:.4f}")

    # Test Classification
    print("\n" + "-"*80)
    print("CLASSIFICATION FORMULATION")
    print("-"*80)

    for act in activations:
        print(f"\nTesting {act.upper()}...")

        if act == 'spline':
            kwargs = {'n_knots': 15, 'init': 'relu'}
        else:
            kwargs = None

        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_classification(
            f'SH + {act.upper()} (cls)', enc,
            coords_train_pop, vals_train_pop,
            coords_test_pop, vals_test_pop,
            verbose=True
        )
        results_exp1.append(res)
        print(f"  Final Acc: {res['accuracy']:.4f}")

    # Summary
    df_exp1 = pd.DataFrame(results_exp1)
    print("\n" + "="*80)
    print("EXPERIMENT 1 SUMMARY")
    print("="*80)
    print(df_exp1[['model', 'task', 'r2', 'accuracy', 'time']].fillna('-').to_string(index=False))
    print("="*80)

    # Analyze regression vs classification
    df_reg = df_exp1[df_exp1['task'] == 'regression']
    df_cls = df_exp1[df_exp1['task'] == 'classification']

    if len(df_reg) >= 2 and len(df_cls) >= 2:
        relu_reg = df_reg[df_reg['model'].str.contains('RELU')]['r2'].values[0]
        spline_reg = df_reg[df_reg['model'].str.contains('SPLINE')]['r2'].values[0]
        relu_cls = df_cls[df_cls['model'].str.contains('RELU')]['accuracy'].values[0]
        spline_cls = df_cls[df_cls['model'].str.contains('SPLINE')]['accuracy'].values[0]

        reg_adv = spline_reg - relu_reg
        cls_adv = spline_cls - relu_cls

        print("\n🎯 KEY FINDING:")
        print(f"Regression:     Spline vs ReLU = {reg_adv:+.4f} ({100*reg_adv/relu_reg:+.2f}%)")
        print(f"Classification: Spline vs ReLU = {cls_adv:+.4f} ({100*cls_adv/relu_cls:+.2f}%)")

        if reg_adv > cls_adv:
            print("\n✅ PAPER PREDICTION CONFIRMED: Spline helps regression more than classification")
        else:
            print("\n❌ PAPER PREDICTION NOT CONFIRMED: No regression advantage")

    # Save results
    df_exp1.to_csv('exp1_regression_vs_classification.csv', index=False)
    print("\n✅ Results saved to exp1_regression_vs_classification.csv")
else:
    print("\n❌ Population data still not available")

EXPERIMENT 1: REGRESSION VS CLASSIFICATION (RERUN)

❌ Population data still not available


---
## Experiment 4: Function Complexity (Total Variation)

Measure complexity of trained models.

In [16]:
print("="*80)
print("EXPERIMENT 4: FUNCTION COMPLEXITY MEASUREMENT")
print("="*80)

def compute_total_variation(model, test_coords, n_paths=500, n_steps=100):
    model.eval()
    tv_values = []
    test_coords_tensor = torch.tensor(test_coords, dtype=torch.float32).to(device)

    for _ in range(n_paths):
        idx = np.random.choice(len(test_coords), 2, replace=False)
        x1 = test_coords_tensor[idx[0]]
        x2 = test_coords_tensor[idx[1]]
        lambdas = torch.linspace(0, 1, n_steps).to(device)
        path = torch.stack([(1-lam)*x1 + lam*x2 for lam in lambdas])
        with torch.no_grad():
            outputs = model(path).cpu().numpy()
        tv = np.sum(np.abs(np.diff(outputs)))
        tv_values.append(tv)

    return np.mean(tv_values)


if 'results_exp1' in locals() and len(results_exp1) > 0:
    print("\nMeasuring complexity for population models...")

    complexity_results = []

    # Measure TV for regression models
    for res in results_exp1:
        if res['task'] == 'regression':
            print(f"\nComputing TV for {res['model']}...")
            model = res['trained_model']
            tv = compute_total_variation(model, coords_test_pop, n_paths=500)
            complexity_results.append({
                'model': res['model'],
                'task': 'population',
                'formulation': 'regression',
                'r2': res['r2'],
                'total_variation': tv
            })
            print(f"  TV = {tv:.2f}, R² = {res['r2']:.4f}")

    # Summary
    df_exp4 = pd.DataFrame(complexity_results)
    print("\n" + "="*80)
    print("EXPERIMENT 4 SUMMARY")
    print("="*80)
    print(df_exp4[['model', 'r2', 'total_variation']].to_string(index=False))
    print("="*80)

    # Check correlation
    if len(df_exp4) >= 3:
        from scipy.stats import pearsonr
        corr, pval = pearsonr(df_exp4['r2'], df_exp4['total_variation'])
        print(f"\nCorrelation (R² vs TV): r = {corr:.3f}, p = {pval:.3f}")
        if pval < 0.05:
            print("✅ Significant correlation found!")
            if corr > 0:
                print("   Higher complexity → better performance")
            else:
                print("   Higher complexity → worse performance (unexpected!)")
        else:
            print("⚠️  Correlation not significant")

    # Save results
    df_exp4.to_csv('exp4_complexity_measurement.csv', index=False)
    print("\n✅ Results saved to exp4_complexity_measurement.csv")
else:
    print("\n⚠️  Exp 1 must complete first to measure complexity")

EXPERIMENT 4: FUNCTION COMPLEXITY MEASUREMENT

⚠️  Exp 1 must complete first to measure complexity


---
## Bonus: Frequency Analysis of Elevation Data

Analyze frequency content to validate "high-frequency" assumption.

In [17]:
print("="*80)
print("FREQUENCY ANALYSIS: Elevation Data")
print("="*80)

# This would require downloading elevation data again
# For now, just note what should be done

print("\nTo complete frequency analysis:")
print("1. Load elevation data (etopo_60s.nc from NB19)")
print("2. Compute 2D FFT of elevation grid")
print("3. Compute power spectrum")
print("4. Compare frequency content at different resolutions")
print("5. Identify if 'high-frequency' assumption holds")
print("\n⚠️  Skipping for now - can add if needed")

FREQUENCY ANALYSIS: Elevation Data

To complete frequency analysis:
1. Load elevation data (etopo_60s.nc from NB19)
2. Compute 2D FFT of elevation grid
3. Compute power spectrum
4. Compare frequency content at different resolutions
5. Identify if 'high-frequency' assumption holds

⚠️  Skipping for now - can add if needed


---
## Summary

In [18]:
print("="*80)
print("NOTEBOOK 19b SUMMARY")
print("="*80)

print("\nCompleted Experiments:")
if 'results_exp1' in locals() and len(results_exp1) > 0:
    print("  ✅ Experiment 1: Regression vs Classification")
    print("  ✅ Experiment 4: Function Complexity (Total Variation)")
else:
    print("  ❌ Experiments failed - check population data loading")

print("\nFiles Generated:")
import os
for f in ['exp1_regression_vs_classification.csv', 'exp4_complexity_measurement.csv']:
    if os.path.exists(f):
        print(f"  ✅ {f}")
    else:
        print(f"  ❌ {f} (not created)")

print("\nNext Steps:")
print("1. Update ANALYSIS_NOTEBOOK19.md with new results")
print("2. Compare Exp 1 results to paper predictions")
print("3. Decide on follow-up experiments based on findings")

print("="*80)

NOTEBOOK 19b SUMMARY

Completed Experiments:
  ❌ Experiments failed - check population data loading

Files Generated:
  ❌ exp1_regression_vs_classification.csv (not created)
  ❌ exp4_complexity_measurement.csv (not created)

Next Steps:
1. Update ANALYSIS_NOTEBOOK19.md with new results
2. Compare Exp 1 results to paper predictions
3. Decide on follow-up experiments based on findings
